# 06 - CNN 1D sobre la curva preprocesada

Modelo en `src/models_cnn.py`: 3 bloques Conv1D+BatchNorm+ReLU+MaxPool, global max pooling y capa densa; pérdida ponderada (pos_weight = negativos/positivos), Adam (lr 1e-3), 20 épocas fijas, batch 64.

Protocolo: CV estratificada de **5 folds x 1 repetición** (la CNN es lenta) y test oficial con IC 95% por bootstrap. Para comparar de forma justa, Random Forest y XGBoost (todas las features) se reevalúan sobre **exactamente los mismos folds**.

Sin data augmentation en esta versión (queda para la entrega final).

In [1]:
import sys, time
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_train_test
from src.preprocessing import preprocess
from src.evaluation import repeated_cv, summarize_cv, report_test
from src.models_cnn import CNNClassifier
from src.models_classic import MODELS, select_columns

pd.set_option('display.width', 200)
Xtr, ytr, Xte, yte = load_train_test()
Ztr, Zte = preprocess(Xtr), preprocess(Xte)
F_train = pd.read_csv('../results/features_train.csv')
F_test = pd.read_csv('../results/features_test.csv')
cols = select_columns(F_train, 'todas')

## 1. Validación cruzada (5 folds x 1) de los tres modelos

In [2]:
resultados, folds_all, oof_all = {}, {}, {}
t0 = time.time()
folds, oof = repeated_cv(lambda: CNNClassifier(epochs=20), Ztr, ytr, n_splits=5, n_repeats=1)
folds_all['cnn_1d'], oof_all['cnn_1d'] = folds, oof
print('CNN: %.0f s' % (time.time() - t0))
for nombre, make in MODELS.items():
    folds, oof = repeated_cv(make, F_train[cols].to_numpy(), ytr, n_splits=5, n_repeats=1)
    folds_all[nombre], oof_all[nombre] = folds, oof
tabla = pd.DataFrame({n: {m: f"{summarize_cv(f).loc[m, 'mean']:.3f} ± {summarize_cv(f).loc[m, 'std']:.3f}"
                         for m in ['recall', 'precision', 'f1', 'pr_auc']} for n, f in folds_all.items()}).T
tabla.to_csv('../results/06_cv_cnn_vs_baseline.csv')
tabla

CNN: 411 s


,recall,precision,f1,pr_auc
cnn_1d,0.557 ± 0.248,0.196 ± 0.173,0.269 ± 0.205,0.270 ± 0.157
random_forest,0.111 ± 0.062,0.800 ± 0.447,0.194 ± 0.109,0.419 ± 0.184
xgboost,0.357 ± 0.241,0.370 ± 0.243,0.338 ± 0.190,0.372 ± 0.181


In [3]:
print('PR-AUC por fold (misma partición para los tres modelos):')
pd.DataFrame({n: f.pr_auc.round(3).to_numpy() for n, f in folds_all.items()})

PR-AUC por fold (misma partición para los tres modelos):


,cnn_1d,random_forest,xgboost
0,0.370,0.456,0.414
1,0.189,0.294,0.229
2,0.079,0.722,0.669
3,0.231,0.270,0.246
4,0.480,0.356,0.300


## 2. Test oficial (5 planetas) con IC 95% por bootstrap

Modelos entrenados con todo train.

In [4]:
cnn = CNNClassifier(epochs=20).fit(Ztr, ytr)
p_cnn = cnn.predict_proba(Zte)[:, 1]
print('== cnn_1d')
print(report_test(yte, p_cnn).round(3), '\n')
np.save('../results/06_cnn_test_scores.npy', p_cnn)
for nombre, make in MODELS.items():
    m = make().fit(F_train[cols].to_numpy(), ytr)
    print('==', nombre)
    print(report_test(yte, m.predict_proba(F_test[cols].to_numpy())[:, 1]).round(3), '\n')

== cnn_1d


           valor  ic95_inf  ic95_sup
recall     1.000     1.000     1.000
precision  0.333     0.238     0.500
f1         0.500     0.385     0.667
pr_auc     0.563     0.372     1.000 



== random_forest


           valor  ic95_inf  ic95_sup
recall     0.400     0.000     0.800
precision  1.000     0.000     1.000
f1         0.571     0.000     0.889
pr_auc     0.580     0.158     1.000 

== xgboost


           valor  ic95_inf  ic95_sup
recall     0.600     0.200     1.000
precision  0.500     0.167     1.000
f1         0.545     0.182     0.889
pr_auc     0.673     0.287     1.000 



## 3. ¿Qué probabilidad le asigna la CNN a cada planeta del test?

In [5]:
idx_pos = np.where(yte == 1)[0]
orden = np.argsort(-p_cnn)
ranking = {i: int(np.where(orden == i)[0][0]) + 1 for i in idx_pos}
print('Probabilidad que la CNN asigna a cada uno de los 5 planetas del test, y su posición en el ranking de 570 estrellas:')
for i in idx_pos:
    print(f'  estrella {i}: prob={p_cnn[i]:.3f}  posición {ranking[i]}/570')
print('Estrellas con prob >= 0.5:', int((p_cnn >= 0.5).sum()))

Probabilidad que la CNN asigna a cada uno de los 5 planetas del test, y su posición en el ranking de 570 estrellas:
  estrella 0: prob=0.869  posición 7/570
  estrella 1: prob=0.974  posición 4/570
  estrella 2: prob=0.890  posición 6/570
  estrella 3: prob=0.985  posición 3/570
  estrella 4: prob=0.967  posición 5/570
Estrellas con prob >= 0.5: 15


## Conclusiones

- **En validación cruzada (37 planetas), la CNN rinde menos que los modelos con features físicas:** PR-AUC 0,27 ± 0,16 contra 0,37 (XGBoost) y 0,42 (Random Forest) sobre los mismos 5 folds. Es esperable: el BLS le entrega a los árboles directamente la información de periodicidad, mientras que la CNN con global max pooling detecta caídas pero no sabe si se repiten.
- **Pero la CNN es la que más planetas detecta:** recall 0,56 (RF 0,11, XGBoost 0,36) a costa de precisión baja (0,20), es decir, muchas falsas alarmas. Los tres modelos usan el umbral 0,5, que no está optimizado.
- **Alta variabilidad entre folds.** El fold 2 es muy malo para la CNN (PR-AUC 0,08, contra 0,67–0,72 de RF y XGBoost) y el fold 4 le favorece (0,48 contra 0,30–0,36). Con ~7 planetas por fold, un par de estrellas cambian mucho el resultado; las diferencias entre modelos deben leerse con cautela.
- **En el test oficial (5 planetas) el orden se invierte:** la CNN ubica los 5 planetas entre las posiciones 3 y 7 de 570 (recall 1,0, PR-AUC 0,56, IC 95% [0,37; 1,00]), comparable a RF (0,58) y XGBoost (0,67). Con 5 positivos y los intervalos solapados, **no se puede afirmar que un modelo sea mejor**; la validación cruzada es la evidencia más confiable.
- **Sin data augmentation ni ajuste de hiperparámetros:** esta es una primera versión (20 épocas fijas, una sola arquitectura y una sola repetición de CV). La brecha con los modelos clásicos puede reducirse con augmentation y con una arquitectura que capture periodicidad; ambos quedan para la entrega final.